## Imports

In [12]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime as datetime

from scipy import stats
from pyhive import presto
from datetime import datetime, timedelta
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import StratifiedKFold

import warnings
warnings.filterwarnings('ignore')

In [13]:
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 300)
pd.set_option('display.max_colwidth', None)

In [14]:
## Connection
connection = presto.connect(
        host='presto-gateway.serving.data.production.internal',
        port=80,
        protocol='http',
        catalog='hive',
        username='manoj.ravirajan@rapido.bike'
)

## Parameter

In [15]:
current_working_directory = os.getcwd()
print(current_working_directory)

/Users/rapido/analytics/customer/Ads-Monetisation/captian_dispatch


In [16]:
local_datasets = '/Users/rapido/local-datasets/customer/ads-monetisation/captian_dispatch_exp/quick_funnel/raw/'

In [17]:
## Parameter 
start_date = '20240301'
end_date = '20240303'
cities = ['Bangalore', 'Jaipur']
service = ['CabEconomy', 'Bike Lite', 'Link', 'Auto']

In [18]:
# Convert the lists to comma-separated strings
city_str = "', '".join(cities)
print(city_str)

service_str = "', '".join(service)
print(service_str)

Bangalore', 'Jaipur
CabEconomy', 'Bike Lite', 'Link', 'Auto


## Datasets

### FE data 

In [20]:
# Loop through each date
current_date = start_date
print('execution started..')
while current_date <= end_date:
    print('running.... ' + current_date)

    query_fare_estimates = f"""

    WITH view AS (
        SELECT 
                yyyymmdd,
                event_props_user_id userId
            FROM 
                clevertap.customer_banner_monetisation_immutable 
            WHERE 
                yyyymmdd = '{current_date}'
                AND event_props_current_city IN ('{city_str}')
                AND event_props_action IN ('view', 'click')
                AND event_props_type = 'GAMBanner'
                AND event_props_sub_type LIKE 'native%'
                AND event_props_screen_name = 'CaptainSearchScreen'
                AND event_props_placement IN ( 'CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd', 'CaptainSearchScreenSlot:d1d2227b-92be-4318-b765-89405bcd1787')
            GROUP BY 1,2
        ),

        fe_tbl AS (
            SELECT
                DATE_FORMAT(DATE_PARSE(fe_tbl.yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
                platform,
                fe_tbl.city,
                fe_tbl.service_name,
                user_id as user_id,
                fare_estimate_id,
                CASE WHEN view.userId IS NOT NULL THEN 'test' ELSE 'control' END group_tc
            FROM
                hive.pricing.fare_estimates_enriched fe_tbl

            LEFT JOIN 
                view
                ON fe_tbl.yyyymmdd = view.yyyymmdd
                AND fe_tbl.user_id = view.userId

            WHERE 
                fe_tbl.yyyymmdd = '{current_date}'
                AND platform IN ('android', 'ios')
                AND fe_tbl.city IN ('{city_str}')
                AND fe_tbl.service_name IN ('{service_str}')
        )

        SELECT 
            *
        FROM 
            fe_tbl
    """ 

    df_fare_estimates = pd.read_sql(query_fare_estimates, connection)
    
    output_filename = f"{current_date}_fare_estimates.csv"    
    df_fare_estimates.to_csv(local_datasets + output_filename, index=False)
    current_date = (pd.to_datetime(current_date) + pd.DateOffset(days=1)).strftime('%Y%m%d')
print('execution completed..')

execution started..
running.... 20240301
running.... 20240302
running.... 20240303
execution completed..


In [21]:
dfs = []

for filename in os.listdir(local_datasets):
    if filename.endswith("_fare_estimates.csv"):
        dfs.append(pd.read_csv(os.path.join(local_datasets, filename)))

df_fare_estimates = pd.concat(dfs, ignore_index=True)
df_fare_estimates.head(5)

,order_date,platform,city,service_name,user_id,fare_estimate_id,group_tc
0,2024-03-01,ios,Bangalore,Link,620393a104809475cba5fa3a,65e1def070181ecd1cc65138,control
1,2024-03-01,android,Jaipur,Link,63ddb021b6d50bf40278b9f4,65e17dcce8a6b332035debd1,control
2,2024-03-01,android,Jaipur,Link,65b608508eea9e2157131d8b,65e16f054eda247b30025f28,control
3,2024-03-01,android,Bangalore,Link,601b013ca6cc7949c1201ad3,65e1ab572e2c7a64abb86520,control
4,2024-03-01,android,Bangalore,Link,658a535a3162a12144f69213,65e179c64f3b0e26e899843a,test


### Orders 

In [22]:
query_order_logs_fact = f"""
    SELECT   
        DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
        city_name AS city,
        service_obj_service_name AS service_name,
        customer_id,
        estimate_id AS fare_estimate_id,
        order_id,
        spd_fraud_flag,
        order_status,
        modified_order_status,
        CASE WHEN modified_order_status in ('OCARA', 'COBRA', 'COBRM') THEN (customer_cancelled_epoch/1000 - order_requested_epoch/1000) END time_spent_in_sec
        
    FROM orders.order_logs_fact ols
    WHERE
        yyyymmdd BETWEEN '{start_date}' 
        AND '{end_date}' 
        AND city_name IN ('{city_str}')
        AND service_obj_service_name IN ('{service_str}')
""" 

df_order_logs_fact = pd.read_sql(query_order_logs_fact, connection)
df_order_logs_fact.to_csv(local_datasets + "order_logs_fact.csv", index=False)
df_order_logs_fact.head(2)

,order_date,city,service_name,customer_id,fare_estimate_id,order_id,spd_fraud_flag,order_status,modified_order_status,time_spent_in_sec
0,2024-03-02,Bangalore,Auto,5dc37462079aed53a6fbb203,65e2a68a227a8425c77d97b8,65e2a6906f2f1e56f188e265,False,dropped,dropped,NaN
1,2024-03-02,Bangalore,Link,641811ecf6a73c9bca900c32,65e2a6c6267cee774184d923,65e2a6d927ce99058891947b,None,customerCancelled,OCARA,475.0


In [23]:
df_order_logs_fact = pd.read_csv(local_datasets + "order_logs_fact.csv")
df_order_logs_fact.head(2)

,order_date,city,service_name,customer_id,fare_estimate_id,order_id,spd_fraud_flag,order_status,modified_order_status,time_spent_in_sec
0,2024-03-02,Bangalore,Auto,5dc37462079aed53a6fbb203,65e2a68a227a8425c77d97b8,65e2a6906f2f1e56f188e265,False,dropped,dropped,NaN
1,2024-03-02,Bangalore,Link,641811ecf6a73c9bca900c32,65e2a6c6267cee774184d923,65e2a6d927ce99058891947b,NaN,customerCancelled,OCARA,475.0


In [24]:
query_banner_monetisation = f"""
    SELECT 
        DATE_FORMAT(DATE_PARSE(yyyymmdd, '%Y%m%d'), '%Y-%m-%d') order_date,
        event_props_action action,
        event_props_current_city city,
        event_props_user_id userId,
        event_props_placement placement,
        event_props_sub_type sub_type,
        event_props_user_id || '-' || event_props_action || '-' || CAST(epoch/1000 AS VARCHAR) AS unique_id        
    FROM 
        raw_internal.clevertap_customer_events_v2 
    WHERE 
        yyyymmdd BETWEEN '{start_date}' 
        AND '{end_date}' 
        AND event_props_current_city IN ('{city_str}')
        AND event_props_type = 'GAMBanner'
        AND event_props_action IN ('view', 'click')
        AND event_props_sub_type LIKE 'native%'
        AND event_props_screen_name = 'CaptainSearchScreen'
        AND event_props_placement IN ( 'CaptainSearchScreenSlot:b99a2cad-d3cf-44f0-b6dd-b7c7cc3cdebd', 'CaptainSearchScreenSlot:d1d2227b-92be-4318-b765-89405bcd1787')
    GROUP BY 1,2,3,4,5,6,7

""" 

df_banner_monetisationt = pd.read_sql(query_banner_monetisation, connection)
df_banner_monetisationt.to_csv(local_datasets + "banner_monetisationt.csv", index=False)
df_banner_monetisationt.head(5)

DatabaseError: {'message': "line 15:13: Column 'event_props_current_city' cannot be resolved", 'errorCode': 47, 'errorName': 'COLUMN_NOT_FOUND', 'errorType': 'USER_ERROR', 'errorLocation': {'lineNumber': 15, 'columnNumber': 13}, 'failureInfo': {'type': 'io.trino.spi.TrinoException', 'message': "line 15:13: Column 'event_props_current_city' cannot be resolved", 'suppressed': [], 'stack': ['io.trino.sql.analyzer.SemanticExceptions.semanticException(SemanticExceptions.java:48)', 'io.trino.sql.analyzer.SemanticExceptions.semanticException(SemanticExceptions.java:43)', 'io.trino.sql.analyzer.SemanticExceptions.missingAttributeException(SemanticExceptions.java:33)', 'io.trino.sql.analyzer.Scope.lambda$resolveField$7(Scope.java:228)', 'java.base/java.util.Optional.orElseThrow(Optional.java:408)', 'io.trino.sql.analyzer.Scope.resolveField(Scope.java:228)', 'io.trino.sql.analyzer.ExpressionAnalyzer$Visitor.visitIdentifier(ExpressionAnalyzer.java:662)', 'io.trino.sql.analyzer.ExpressionAnalyzer$

In [ ]:
df_banner_monetisationt = pd.read_csv(local_datasets + "banner_monetisationt.csv")
df_banner_monetisationt.head(2)

### Merge

In [ ]:
result_df = pd.merge(df_fare_estimates, df_order_logs_fact, 
                     how='left', 
                     on=['order_date', 'fare_estimate_id'],
                     suffixes=('_fe', '_rr_net'))
result_df = result_df[['order_date', 'platform', 'city_fe', 'service_name_fe', 'user_id', 
                       'fare_estimate_id', 'group_tc', 'customer_id', 'order_id', 'spd_fraud_flag', 
                       'order_status', 'modified_order_status', 'time_spent_in_sec']]
result_df.head(2)

In [ ]:
result_df1 = pd.merge(result_df, df_banner_monetisationt,
                      how='left', 
                      left_on=['order_date', 'user_id'],
                      right_on=['order_date', 'userId'])
result_df1 = result_df1[['order_date', 'platform', 'city_fe', 'service_name_fe', 'user_id',
                         'fare_estimate_id', 'group_tc', 'customer_id', 'order_id', 'spd_fraud_flag', 
                         'order_status', 'modified_order_status', 'time_spent_in_sec',
                         'action', 'userId', 'placement', 'sub_type','hhmmss', 'unique_id']]
result_df1.head(2)

In [22]:
result_df1['view'] = result_df1.apply(lambda row: row['unique_id'] if row['action'] == 'view' else None, axis=1)
result_df1['click'] = result_df1.apply(lambda row: row['unique_id'] if row['action'] == 'click' else None, axis=1)

In [23]:
result_df1.head(2)

,order_date,platform,city_fe,service_name_fe,user_id,fare_estimate_id,group_tc,customer_id,order_id,spd_fraud_flag,order_status,modified_order_status,time_spent_in_sec,action,userId,placement,sub_type,hhmmss,unique_id,view,click
0,2024-02-12,ios,Bangalore,Link,65c99144045fd0124b617eaf,65c994a2d7f9fa3132dc2538,control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
1,2024-02-12,android,Bangalore,Link,65ca0471656f1561195874b1,65ca04ac4e6f81663d891ade,control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
2,2024-02-12,android,Jaipur,Auto,65823476e7f7653da43ed435,65c96137d7f9fac192d8e82e,control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
3,2024-02-12,android,Bangalore,Auto,656822e0ba745c5824a75acb,65c9955079fd1339db25fa40,control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
4,2024-02-12,ios,Bangalore,Auto,617ea8c8544f127f1a4877c8,65c9fe0f79fd1307422f06ca,control,617ea8c8544f127f1a4877c8,65c9fe3034672646ae5187df,False,dropped,dropped,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
23586396,2024-02-13,android,Bangalore,Link,6053855bc107338c9f796378,65caf3d2134f6d35dc6f6d5c,control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
23586397,2024-02-13,ios,Bangalore,CabEconomy,61966279090a2b06643f2b60,65cba9f6522019b03c49e3fc,control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
23586398,2024-02-13,ios,Jaipur,Auto,62d2afe7b53fa3b8442cc33e,65cb2986e7bf3d7a9e9b789d,control,62d2afe7b53fa3b8442cc33e,65cb298a62f6d841db389d9e,False,dropped,dropped,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None
23586399,2024-02-13,android,Bangalore,Auto,63f2029300965c19ee42ef64,65cb49583da7acebc455dc46,control,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,None,None


In [29]:
aggregated_df = result_df1.groupby(['order_date', 'platform', 'city_fe', 'group_tc'])\
.agg(
    users_count=('user_id', 'nunique'),
    fe_count=('fare_estimate_id', 'nunique'),
    rr_count=('order_id', 'nunique'),
    net_count=('order_id', lambda x: x[(result_df1['order_status'] == 'dropped') 
                                       & 
                                       (result_df1['spd_fraud_flag'] != True)].nunique()),
    view_count=('view', 'nunique'),
    click_count=('click', 'nunique'),
    
    cobrm=('order_id', lambda x: x[(result_df1['modified_order_status'] == 'COBRM')].nunique()),
    cobra=('order_id', lambda x: x[(result_df1['modified_order_status'] == 'COBRA')].nunique()),
    ocara=('order_id', lambda x: x[(result_df1['modified_order_status'] == 'OCARA')].nunique()),
    
    median_cobrm_time=('time_spent_in_sec', lambda x: x[result_df1['modified_order_status'] == 'COBRM'].quantile(0.50) if any(result_df1['modified_order_status'] == 'COBRM') else None),
    median_cobra_time=('time_spent_in_sec', lambda x: x[result_df1['modified_order_status'] == 'COBRA'].quantile(0.50) if any(result_df1['modified_order_status'] == 'COBRA') else None)
)\
.reset_index()

In [30]:
aggregated_df

,order_date,platform,city_fe,group_tc,users_count,fe_count,rr_count,net_count,view_count,click_count,cobrm,cobra,ocara,median_cobrm_time,median_cobra_time
0,2024-02-12,android,Bangalore,control,279308,1185189,405150,191932,0,0,1826,103122,69018,104.0,125.0
1,2024-02-12,android,Bangalore,test,1196,8719,3748,1628,1200,11,31,967,794,92.0,120.0
2,2024-02-12,android,Jaipur,control,56224,189479,54240,28521,0,0,372,8982,13229,127.0,128.0
3,2024-02-12,android,Jaipur,test,1194,6826,2896,1243,1197,26,44,630,775,118.0,129.0
4,2024-02-12,ios,Bangalore,control,72590,302274,119642,54216,0,0,291,29048,20659,103.0,122.0
5,2024-02-12,ios,Jaipur,control,6839,26357,8315,4279,0,0,56,1442,2011,93.0,98.0
6,2024-02-13,android,Bangalore,control,264133,993239,374991,184595,0,0,1573,88757,63368,112.0,125.0
7,2024-02-13,android,Bangalore,test,3600,21344,10165,4122,3608,39,162,2590,2489,106.0,114.0
8,2024-02-13,android,Jaipur,control,49797,178302,44953,25184,0,0,248,6746,10350,122.0,131.0
9,2024-02-13,android,Jaipur,test,3669,18998,7766,3370,3689,51,146,1556,2064,110.0,128.0


In [32]:
aggregated_df.to_clipboard(index=False)

In [33]:
aggregated_df1 = result_df1.groupby(['platform', 'city_fe', 'group_tc'])\
.agg(
    users_count=('user_id', 'nunique'),
    fe_count=('fare_estimate_id', 'nunique'),
    rr_count=('order_id', 'nunique'),
    net_count=('order_id', lambda x: x[(result_df1['order_status'] == 'dropped') 
                                       & 
                                       (result_df1['spd_fraud_flag'] != True)].nunique()),
    view_count=('view', 'nunique'),
    click_count=('click', 'nunique'),
    
    cobrm=('order_id', lambda x: x[(result_df1['modified_order_status'] == 'COBRM')].nunique()),
    cobra=('order_id', lambda x: x[(result_df1['modified_order_status'] == 'COBRA')].nunique()),
    ocara=('order_id', lambda x: x[(result_df1['modified_order_status'] == 'OCARA')].nunique()),
    
    median_cobrm_time=('time_spent_in_sec', lambda x: x[result_df1['modified_order_status'] == 'COBRM'].quantile(0.50) if any(result_df1['modified_order_status'] == 'COBRM') else None),
    median_cobra_time=('time_spent_in_sec', lambda x: x[result_df1['modified_order_status'] == 'COBRA'].quantile(0.50) if any(result_df1['modified_order_status'] == 'COBRA') else None)
)\
.reset_index()

In [34]:
aggregated_df1

,platform,city_fe,group_tc,users_count,fe_count,rr_count,net_count,view_count,click_count,cobrm,cobra,ocara,median_cobrm_time,median_cobra_time
0,android,Bangalore,control,819310,5034789,1887400,932381,0,0,8485,437575,327302,107.0,123.0
1,android,Bangalore,test,29426,149894,73239,34027,34231,297,663,17549,14233,110.0,115.0
2,android,Jaipur,control,192063,927279,258853,134595,0,0,1898,43841,61780,129.0,132.0
3,android,Jaipur,test,11678,60987,26751,12249,13195,188,396,5729,6379,117.0,129.0
4,ios,Bangalore,control,200408,1508276,603483,280348,0,0,1352,135940,107656,99.0,119.0
5,ios,Bangalore,test,5,17,8,4,5,0,0,1,2,NaN,117.0
6,ios,Jaipur,control,22680,128167,41533,21529,0,0,252,7538,9382,109.0,104.0
7,ios,Jaipur,test,3,22,2,2,3,0,0,0,0,NaN,NaN


In [35]:
aggregated_df1.to_clipboard(index=False)

In [36]:
aggregated_df2 = result_df1.groupby(['platform', 'city_fe', 'service_name_fe', 'group_tc'])\
.agg(
    users_count=('user_id', 'nunique'),
    fe_count=('fare_estimate_id', 'nunique'),
    rr_count=('order_id', 'nunique'),
    net_count=('order_id', lambda x: x[(result_df1['order_status'] == 'dropped') 
                                       & 
                                       (result_df1['spd_fraud_flag'] != True)].nunique()),
    view_count=('view', 'nunique'),
    click_count=('click', 'nunique'),
    
    cobrm=('order_id', lambda x: x[(result_df1['modified_order_status'] == 'COBRM')].nunique()),
    cobra=('order_id', lambda x: x[(result_df1['modified_order_status'] == 'COBRA')].nunique()),
    ocara=('order_id', lambda x: x[(result_df1['modified_order_status'] == 'OCARA')].nunique()),
    
    median_cobrm_time=('time_spent_in_sec', lambda x: x[result_df1['modified_order_status'] == 'COBRM'].quantile(0.50) if any(result_df1['modified_order_status'] == 'COBRM') else None),
    median_cobra_time=('time_spent_in_sec', lambda x: x[result_df1['modified_order_status'] == 'COBRA'].quantile(0.50) if any(result_df1['modified_order_status'] == 'COBRA') else None)
)\
.reset_index()

In [37]:
aggregated_df2

,platform,city_fe,service_name_fe,group_tc,users_count,fe_count,rr_count,net_count,view_count,click_count,cobrm,cobra,ocara,median_cobrm_time,median_cobra_time
0,android,Bangalore,Auto,control,819181,4806117,1694998,746969,0,0,8485,437575,320644,107.0,123.0
1,android,Bangalore,Auto,test,29419,141706,66341,27410,34224,297,663,17549,13960,110.0,115.0
2,android,Bangalore,Bike Lite,control,155877,502609,168480,84551,0,0,849,35591,35813,106.0,114.0
3,android,Bangalore,Bike Lite,test,5271,15003,7016,3188,6115,39,52,1768,1408,105.0,113.0
4,android,Bangalore,CabEconomy,control,817185,4711387,1628251,686685,0,0,8451,436438,316068,107.0,123.0
5,android,Bangalore,CabEconomy,test,29379,138716,64346,25654,34174,295,658,17517,13782,110.0,115.0
6,android,Bangalore,Link,control,819246,4887626,1766524,817381,0,0,8485,437573,321702,107.0,123.0
7,android,Bangalore,Link,test,29420,144522,68982,30033,34223,297,663,17549,13982,110.0,115.0
8,android,Jaipur,Auto,control,192040,886522,223695,101041,0,0,1898,43841,60236,130.0,131.0
9,android,Jaipur,Auto,test,11677,57470,23690,9318,13194,188,396,5729,6259,118.0,129.0


In [39]:
aggregated_df2.to_clipboard(index=False)